In [1]:
import os
import pandas as pd

DATA_DIR = r"C:\Users\1\Documents\GitHub\ecommerce-olist-analysis\data"

for file in sorted(os.listdir(DATA_DIR)):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_DIR, file), encoding="utf-8-sig")
        print(f"{file}: {df.shape[0]} строк, {df.shape[1]} столбцов")
        print("   ", list(df.columns))

olist_closed_deals_dataset.csv: 842 строк, 14 столбцов
    ['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue']
olist_customers_dataset.csv: 99441 строк, 5 столбцов
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
olist_geolocation_dataset.csv: 1000163 строк, 5 столбцов
    ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
olist_marketing_qualified_leads_dataset.csv: 8000 строк, 4 столбцов
    ['mql_id', 'first_contact_date', 'landing_page_id', 'origin']
olist_order_items_dataset.csv: 112650 строк, 7 столбцов
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
olist_order_payments_dataset.csv: 103886 строк, 5 столбцов
    ['order_id',

In [2]:
import psycopg2
from getpass import getpass

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="olist",
    user="postgres",
    password=getpass("Пароль postgres: ")
)
cur = conn.cursor()

# Порядок важен: родительские таблицы раньше дочерних
tables = [
    ("customers",                         "olist_customers_dataset.csv"),
    ("sellers",                           "olist_sellers_dataset.csv"),
    ("product_category_name_translation", "product_category_name_translation.csv"),
    ("products",                          "olist_products_dataset.csv"),
    ("orders",                            "olist_orders_dataset.csv"),
    ("order_items",                       "olist_order_items_dataset.csv"),
    ("order_payments",                    "olist_order_payments_dataset.csv"),
    ("order_reviews",                     "olist_order_reviews_dataset.csv"),
    ("geolocation",                       "olist_geolocation_dataset.csv"),
]

for table, file in tables:
    with open(os.path.join(DATA_DIR, file), encoding="utf-8-sig") as f:
        cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true)", f)
    conn.commit()                      # зафиксировать загрузку
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]} строк")

cur.close()
conn.close()

Пароль postgres:  ········


customers: 99441 строк
sellers: 3095 строк
product_category_name_translation: 71 строк
products: 32951 строк
orders: 99441 строк
order_items: 112650 строк
order_payments: 103886 строк
order_reviews: 99224 строк
geolocation: 1000163 строк


In [2]:
conn = psycopg2.connect(host="localhost", port=5432, dbname="olist",
                        user="postgres", password=getpass("Пароль postgres: "))
cur = conn.cursor()

# Воронка продавцов: сначала лиды (родитель), потом сделки
tables_mkt = [
    ("marketing_qualified_leads", "olist_marketing_qualified_leads_dataset.csv"),
    ("closed_deals",              "olist_closed_deals_dataset.csv"),
]

for table, file in tables_mkt:
    with open(os.path.join(DATA_DIR, file), encoding="utf-8-sig") as f:
        cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true)", f)
    conn.commit()
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]} строк")

cur.close()
conn.close()

TypeError: 'module' object is not callable. Did you mean: 'getpass.getpass(...)'?